# Dataset setup

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
!mkdir /content/datasets

In [ ]:
# This should take 1-2 minutes
# It unzips the dataset in the runtime's local SSD, so when
# you disconnect, it gets deleted
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

# Data loading

In [ ]:
import torch
from pathlib import Path

from torchvision.datasets import CelebA

In [ ]:
# Do *not* put `celeba` in the path.
# The dataset class will do that automatically!
data_root = Path("/content/datasets")

In [ ]:
celeba = CelebA(root=data_root, split="test", download=False)

In [ ]:
# This should be 19,962
len(celeba)

# Embedding

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load CLIP model and processor from HuggingFace
# Reference: https://huggingface.co/openai/clip-vit-base-patch32
MODEL_NAME = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f"Model loaded: {MODEL_NAME}")

In [ ]:
class CelebAImageDataset(Dataset):
    """
    Lightweight wrapper around CelebA that returns only the PIL image.
    The CLIPProcessor will handle all preprocessing (resize, crop, normalize).
    """
    def __init__(self, celeba_dataset):
        self.dataset = celeba_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, _ = self.dataset[idx]  # discard attributes
        return image  # PIL Image


def collate_fn(images):
    """Batch PIL images using CLIPProcessor."""
    return processor(images=images, return_tensors="pt", padding=True)


image_dataset = CelebAImageDataset(celeba)
dataloader = DataLoader(
    image_dataset,
    batch_size=256,
    shuffle=False,       # keep original order so index = celeba index
    num_workers=2,
    collate_fn=collate_fn,
)

print(f"Dataset size : {len(image_dataset)} images")
print(f"Num batches  : {len(dataloader)} (batch_size=256)")

In [ ]:
all_embeddings = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Embedding CelebA"):
        pixel_values = batch["pixel_values"].to(device)

        # Step 1: passa le immagini al vision encoder (ViT)
        #         restituisce un oggetto BaseModelOutputWithPooling
        vision_outputs = model.vision_model(pixel_values=pixel_values)

        # Step 2: estrai il pooler_output -> tensore (batch_size, 768)
        #         è la rappresentazione del token [CLS] dopo il transformer
        pooled = vision_outputs.pooler_output

        # Step 3: applica il projection layer -> tensore (batch_size, 512)
        #         mappa da 768 (dimensione interna ViT) a 512 (spazio CLIP condiviso)
        image_features = model.visual_projection(pooled)

        # Step 4: L2-normalize
        image_features = F.normalize(image_features, p=2, dim=-1)

        all_embeddings.append(image_features.cpu())

embeddings = torch.cat(all_embeddings, dim=0)
print(f"Embeddings shape: {embeddings.shape}")  # atteso: (19962, 512)

In [ ]:
save_path = "/content/drive/MyDrive/datasets/celeba_clip_embeddings.pt"

torch.save({
    "embeddings": embeddings,   # shape (19962, 512), float32, L2-normalized
    "model": MODEL_NAME,        # track which model generated these
    "split": "test",
}, save_path)

print(f"Saved to: {save_path}")
print(f"File size: {embeddings.element_size() * embeddings.nelement() / 1e6:.1f} MB")

# Metrics

In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
):
    """
    Evaluate the retrieval performance for a single source image.

    Args:
    ----
        retrieved_indices: list of image IDs predicted by the model,
            ordered by similarity (descending).
        ground_truth_indices: list of valid target IDs from the benchmark JSON.
        k: the cutoff for top-K evaluation (e.g., 1, 5, 10).

    Return:
    ------
        A dictionary containing Recall@K and Precision@K.

    """
    # Isolate the top K predictions
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

In [ ]:
# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

# Evaluation

In [ ]:
import json

In [ ]:
annotations_path = Path("/content/drive/MyDrive/datasets/celeba_evaluation.json")

with open(annotations_path, "r") as f:
    annotations = json.load(f)

len(annotations)

In [ ]:
# `annotations` is a list of queries
# Each query is a dictionary with these keys:
# - `query`: the textual query itself
# - `ground_truth`: a dictionary of images
#
# Each element in `ground_truth` is structured as:
# {
#    idx: list[int]
# }
# `idx` is the source image that you have to use together with the `query`
# `list[int]` is the list of acceptable target images, i.e., the images
# that you should retrieve

print(annotations[0].keys())
print()
print("Query:", annotations[0]["query"])
print("Source images:", len(annotations[0]["ground_truth"].keys()))

Let's test the evaluation function. Let's simulate retrieving data for the first image / query.

In [ ]:
print("Nothing retrieved:\n", evaluate_retrieval([], annotations[0]["ground_truth"]["13"], 1))
print()
print("Retrieved only one wrong image:\n", evaluate_retrieval([0], annotations[0]["ground_truth"]["13"], 1))
print()
print("Retrieved 10 correct images:\n", evaluate_retrieval(annotations[0]["ground_truth"]["13"][:10], annotations[0]["ground_truth"]["13"], 1))
print()
# This returns 0 and 0 because top-k is set to 1, and the first 5 images
# (by highest similarity) are incorrect. We need to jump at least to top-6 here!
print("Retrieved 5 correct images and 5 wrong images:\n", evaluate_retrieval([0, 1, 2, 3, 4] + annotations[0]["ground_truth"]["13"][:5], annotations[0]["ground_truth"]["13"], 1))
print("Retrieved 5 correct images and 5 wrong images:\n", evaluate_retrieval([0, 1, 2, 3, 4] + annotations[0]["ground_truth"]["13"][:5], annotations[0]["ground_truth"]["13"], 6))

In [ ]:
# This is our source image
# Basically, the `key` in the provided JSON corresponds to the
# image index. Note, however, that the key is in string format,
# so remember to convert it with `int(key)`!
celeba[13][0]

In [ ]:
# This is one of the accepted matches
celeba[annotations[0]["ground_truth"]["13"][0]][0]

In [ ]:
# This is another one of the accepted matches
celeba[annotations[0]["ground_truth"]["13"][1]][0]

In [ ]:
# And yet another one of the accepted matches
celeba[annotations[0]["ground_truth"]["13"][2]][0]

As you can see, all these images are pretty similar to the first one (the "source").
This is exactly what we want.
We constructed the ground truth by taking images that match the given query, with a Hamming distance of at most 2 from the source (i.e., having a perfect match for the query is impossible, so we allow some small variability on other attributes, too).

# Misc

In [ ]:
# Assign a unique index to each attribute, and get the inverse mapping
idx2attribute = {idx: name for idx, name in enumerate(celeba.attr_names)}
attribute2idx = {name: idx for idx, name in enumerate(celeba.attr_names)}